# Ethereum AML GNN Visualization Pipeline (Production)

**Purpose:** Generate five complementary visualizations from trained NodeGINe GNN model on Ethereum OFAC sanctions detection task.

**Data:** Real OFAC-sanctioned Ethereum addresses + EOA-only transaction graph (~889K nodes, ~23M edges)

**Output:** Publication-ready figures demonstrating:
1. UMAP embedding projections (representation quality)
2. Ego-centric subgraph motifs (transaction patterns)
3. Temporal evolution snapshots (temporal dynamics)
4. SHAP explainability (model decisions)
5. Evaluation metrics (confusion matrix, ROC/PR curves)

---

## Cell 1: Install Dependencies & Imports

In [ ]:
# Install required packages (Colab-compatible)
import subprocess
import sys

packages = [
    'torch',
    'torch-geometric',
    'pyg-lib',
    'umap-learn',
    'pandas',
    'pyarrow',
    'networkx',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'pyvis',
    'shap'
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ All dependencies installed")

In [ ]:
# Core imports
import os
import sys
import logging
import warnings
from pathlib import Path
from typing import Tuple, Dict, List, Optional
import json
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GINEConv, BatchNorm, Linear

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc, precision_recall_curve,
    f1_score, precision_score, recall_score
)
import networkx as nx
from pyvis.network import Network
import umap
import shap

# Suppress warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Matplotlib settings
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
sns.set_style('whitegrid')

print("✓ All imports loaded")

## Cell 2: Configuration & Logging

In [ ]:
# Setup logging
def setup_logger(name: str) -> logging.Logger:
    """Configure logger with console output."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)
    
    # Remove existing handlers to avoid duplicates
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)
    
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_format = logging.Formatter(
        '%(asctime)s [%(levelname)s] %(name)s: %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    console_handler.setFormatter(console_format)
    logger.addHandler(console_handler)
    return logger

logger = setup_logger('AML_Visualizations_Production')

# Configuration
class Config:
    """Data paths and hyperparameters."""
    
    # Base path for your project within Google Drive
    BASE_DRIVE_PATH = Path("/content/drive/MyDrive/Math_168_Folder")

    # Data directories
    DATA_DIR = BASE_DRIVE_PATH / "data" / "aggressively_processed"
    TRAINING_RESULTS_DIR = BASE_DRIVE_PATH / "training_results"
    OUTPUT_DIR = BASE_DRIVE_PATH / "Data-Visualization" / "outputs"

    # Input files (REAL DATA ONLY)
    FORMATTED_TRANSACTIONS = DATA_DIR / "formatted_transactions.parquet"
    NODE_LABELS = DATA_DIR / "node_labels.parquet"
    DATA_SPLITS = DATA_DIR / "data_splits.json"
    BEST_MODEL_CHECKPOINT = TRAINING_RESULTS_DIR / "best_model.pt"
    
    # Visualization hyperparameters
    UMAP_NEIGHBORS = 15
    UMAP_MIN_DIST = 0.1
    UMAP_N_EPOCHS = 200
    
    EMBEDDING_SUBSAMPLE_SIZE = 2000  # For visualization clarity
    EGO_SUBGRAPH_RADIUS = 2
    EGO_SUBGRAPH_MAX_SAMPLES = 4
    TEMPORAL_WINDOWS = 10
    
    SHAP_SAMPLE_SIZE = 100
    SHAP_BACKGROUND_SIZE = 50
    
    # Output settings
    DPI = 300
    FIGURE_FORMAT = 'png'
    
    @classmethod
    def validate(cls) -> bool:
        """Validate required files exist."""
        required = [cls.FORMATTED_TRANSACTIONS, cls.NODE_LABELS, cls.DATA_SPLITS, cls.BEST_MODEL_CHECKPOINT]
        missing = [f for f in required if not f.exists()]
        if missing:
            logger.error(f"Missing files: {missing}")
            return False
        cls.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        logger.info(f"✓ All files found. Output: {cls.OUTPUT_DIR}")
        return True

config = Config()
if not config.validate():
    raise RuntimeError("Configuration validation failed.")

## Cell 3: Model Architecture

In [ ]:
class NodeGINe(nn.Module):
    """GINe adapted for node-level classification."""
    
    def __init__(self, num_features, num_gnn_layers, n_classes=2,
                 n_hidden=100, edge_updates=False, residual=True,
                 edge_dim=None, dropout=0.0, final_dropout=0.5):
        super().__init__()
        self.n_hidden = n_hidden
        self.num_gnn_layers = num_gnn_layers
        self.edge_updates = edge_updates

        self.node_emb = nn.Linear(num_features, n_hidden)
        self.edge_emb = nn.Linear(edge_dim, n_hidden)

        self.convs = nn.ModuleList()
        self.emlps = nn.ModuleList()
        self.batch_norms = nn.ModuleList()

        for _ in range(self.num_gnn_layers):
            conv = GINEConv(nn.Sequential(
                nn.Linear(n_hidden, n_hidden),
                nn.ReLU(),
                nn.Linear(n_hidden, n_hidden)
            ), edge_dim=n_hidden)
            if self.edge_updates:
                self.emlps.append(nn.Sequential(
                    nn.Linear(3 * n_hidden, n_hidden),
                    nn.ReLU(),
                    nn.Linear(n_hidden, n_hidden),
                ))
            self.convs.append(conv)
            self.batch_norms.append(BatchNorm(n_hidden))

        self.mlp = nn.Sequential(
            Linear(n_hidden, 50), nn.ReLU(), nn.Dropout(final_dropout),
            Linear(50, 25), nn.ReLU(), nn.Dropout(final_dropout),
            Linear(25, n_classes)
        )

    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)

        for i in range(self.num_gnn_layers):
            x = (x + F.relu(self.batch_norms[i](self.convs[i](x, edge_index, edge_attr)))) / 2
            if self.edge_updates:
                edge_attr = edge_attr + self.emlps[i](
                    torch.cat([x[src], x[dst], edge_attr], dim=-1)
                ) / 2

        return self.mlp(x)

    def get_embeddings(self, x, edge_index, edge_attr):
        """Extract node embeddings before MLP head."""
        src, dst = edge_index
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)

        for i in range(self.num_gnn_layers):
            x = (x + F.relu(self.batch_norms[i](self.convs[i](x, edge_index, edge_attr)))) / 2
            if self.edge_updates:
                edge_attr = edge_attr + self.emlps[i](
                    torch.cat([x[src], x[dst], edge_attr], dim=-1)
                ) / 2

        return x

logger.info("✓ NodeGINe model class loaded")

## Cell 4: Data Loading & Preparation

In [ ]:
def z_norm(data):
    """Z-score normalization."""
    std = data.std(0).unsqueeze(0)
    std = torch.where(std == 0, torch.tensor(1.0), std)
    return (data - data.mean(0).unsqueeze(0)) / std

def build_pyg_data(df_edges, df_labels, data_splits):
    """Build PyG Data object from real formatted data."""
    logger.info("Building PyG Data object from real data...")

    src = torch.LongTensor(df_edges['from_id'].values)
    dst = torch.LongTensor(df_edges['to_id'].values)
    edge_index = torch.stack([src, dst], dim=0)

    edge_feat_cols = ['EdgeID', 'Timestamp', 'Amount Sent', 'Sent Currency',
                      'Amount Received', 'Received Currency', 'Payment Format', 'Is Laundering']
    edge_feat_df = df_edges[edge_feat_cols].copy()
    edge_attr = torch.tensor(edge_feat_df.values, dtype=torch.float32)

    max_node_id = max(int(src.max()), int(dst.max())) + 1
    x = torch.ones(max_node_id, 1, dtype=torch.float32)

    y = torch.zeros(max_node_id, dtype=torch.long)
    for nid, lbl in zip(df_labels['node_id'].values, df_labels['is_sanctioned'].values):
        if nid < max_node_id:
            y[int(nid)] = int(lbl)

    train_mask = torch.zeros(max_node_id, dtype=torch.bool)
    val_mask = torch.zeros(max_node_id, dtype=torch.bool)

    if 'train_edge_ids' in data_splits:
        train_edge_ids = data_splits['train_edge_ids']
        train_edges_df = df_edges.iloc[train_edge_ids]
        train_nodes = set(train_edges_df['from_id'].values) | set(train_edges_df['to_id'].values)
        for nid in train_nodes:
            if nid < max_node_id:
                train_mask[int(nid)] = True

    if 'val_edge_ids' in data_splits:
        val_edge_ids = data_splits['val_edge_ids']
        val_edges_df = df_edges.iloc[val_edge_ids]
        val_nodes = set(val_edges_df['from_id'].values) | set(val_edges_df['to_id'].values)
        for nid in val_nodes:
            if nid < max_node_id:
                val_mask[int(nid)] = True
        val_mask = val_mask & ~train_mask

    edge_attr = z_norm(edge_attr)
    x = z_norm(x)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y,
                train_mask=train_mask, val_mask=val_mask)

    logger.info(f"  Nodes: {data.num_nodes:,}, Edges: {data.num_edges:,}")
    logger.info(f"  Sanctioned: {(data.y == 1).sum().item()}, "
                f"Train: {data.train_mask.sum().item():,}, "
                f"Val: {data.val_mask.sum().item():,}")

    return data

# Load real data
logger.info("[LOADING DATA]")
df_edges = pd.read_parquet(config.FORMATTED_TRANSACTIONS)
df_labels = pd.read_parquet(config.NODE_LABELS)
with open(config.DATA_SPLITS) as f:
    data_splits = json.load(f)

logger.info(f"  Loaded {len(df_edges):,} edges")
logger.info(f"  Loaded {len(df_labels):,} node labels")
logger.info(f"  Sanctioned nodes: {(df_labels['is_sanctioned'] == 1).sum()}")

# Build PyG Data
data = build_pyg_data(df_edges, df_labels, data_splits)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)
logger.info(f"✓ Device: {device}")

## Cell 5: Load Trained Model

In [ ]:
logger.info("[LOADING MODEL]")
checkpoint = torch.load(config.BEST_MODEL_CHECKPOINT, map_location=device)

model = NodeGINe(
    num_features=1,
    num_gnn_layers=2,
    n_classes=2,
    n_hidden=66,
    edge_updates=False,
    edge_dim=8,
    dropout=0.01,
    final_dropout=0.1
)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

logger.info(f"  Loaded from epoch {checkpoint['epoch'] + 1}")
logger.info(f"  Best Val F1: {checkpoint['val_f1']:.4f}")
logger.info(f"✓ Model loaded and ready")

## Visualization 1: UMAP Embeddings (Production)

In [ ]:
logger.info("\n[VIZ 1] UMAP Embedding Projection")

# Extract embeddings from validation set
logger.info("  Extracting node embeddings from trained model...")
with torch.no_grad():
    embeddings_full = model.get_embeddings(
        data.x, data.edge_index, data.edge_attr
    ).cpu().numpy()

# Subsample validation nodes for visualization
val_indices = torch.nonzero(data.val_mask, as_tuple=True)[0].numpy()
sample_indices = np.random.choice(len(val_indices),
                                  min(config.EMBEDDING_SUBSAMPLE_SIZE, len(val_indices)),
                                  replace=False)
sample_node_ids = val_indices[sample_indices]

embeddings = embeddings_full[sample_node_ids]
labels = data.y[sample_node_ids].cpu().numpy()

logger.info(f"  Sampled {len(sample_node_ids)} validation nodes")

# UMAP projection
logger.info("  Computing UMAP projection...")
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

reducer = umap.UMAP(
    n_neighbors=config.UMAP_NEIGHBORS,
    min_dist=config.UMAP_MIN_DIST,
    metric='euclidean',
    n_epochs=config.UMAP_N_EPOCHS,
    random_state=42,
    verbose=1
)
embeddings_2d = reducer.fit_transform(embeddings_scaled)

# Plot
fig, ax = plt.subplots(figsize=(14, 11))

mask_benign = labels == 0
ax.scatter(embeddings_2d[mask_benign, 0], embeddings_2d[mask_benign, 1],
           c='steelblue', alpha=0.4, s=30, label=f'Non-sanctioned (n={mask_benign.sum()})')

mask_sanctioned = labels == 1
ax.scatter(embeddings_2d[mask_sanctioned, 0], embeddings_2d[mask_sanctioned, 1],
           c='darkred', alpha=0.95, s=250, marker='*',
           label=f'OFAC-sanctioned (n={mask_sanctioned.sum()})',
           edgecolors='darkred', linewidths=2)

ax.set_xlabel('UMAP Dimension 1', fontsize=13, fontweight='bold')
ax.set_ylabel('UMAP Dimension 2', fontsize=13, fontweight='bold')
ax.set_title('Node Embedding Projections: GNN Learned Representations\n' +
             '(UMAP 2D projection, validation node subsample)',
             fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.2, linestyle='--')

plt.tight_layout()
output_path = config.OUTPUT_DIR / 'viz1_embedding_projection_umap.png'
fig.savefig(output_path, dpi=config.DPI, bbox_inches='tight')
logger.info(f"  ✓ Saved: {output_path}")
plt.close()

## Visualization 2: Ego-Centric Subgraphs (Production)

In [ ]:
logger.info("\n[VIZ 2] Ego-Centric Subgraph Motifs")

def extract_ego_subgraph_df(df_edges: pd.DataFrame, center_node: int, radius: int = 2) -> pd.DataFrame:
    """
    Extract ego subgraph directly from DataFrame (memory-efficient).
    
    Uses BFS to find all nodes within radius hops from center_node,
    then filters edges to those within the ego subgraph.
    
    Args:
        df_edges: DataFrame with 'from_id' and 'to_id' columns
        center_node: Central node ID
        radius: Hop distance for ego subgraph
    
    Returns:
        DataFrame of edges in the ego subgraph
    """
    visited = {center_node}
    current_level = {center_node}
    
    # BFS to find all nodes within radius hops
    for hop in range(radius):
        next_level = set()
        
        # Find edges touching current level (both directions)
        edges_from = df_edges[df_edges['from_id'].isin(current_level)]
        edges_to = df_edges[df_edges['to_id'].isin(current_level)]
        
        # Collect nodes on other end
        next_level.update(edges_from['to_id'].values)
        next_level.update(edges_to['from_id'].values)
        
        # Only keep new nodes
        next_level -= visited
        current_level = next_level
        visited.update(next_level)
    
    # Extract edges where both endpoints are in ego subgraph
    ego_edges = df_edges[
        (df_edges['from_id'].isin(visited)) & 
        (df_edges['to_id'].isin(visited))
    ].copy()
    
    return ego_edges, visited

# Get sanctioned nodes
sanctioned_nodes = df_labels[df_labels['is_sanctioned'] == 1]['node_id'].values
logger.info(f"  Found {len(sanctioned_nodes)} sanctioned nodes")

# Extract and visualize ego subgraphs (memory-efficient)
logger.info(f"  Extracting {config.EGO_SUBGRAPH_MAX_SAMPLES} ego subgraphs directly from DataFrame...")
ego_graphs = {}
ego_node_sets = {}

for sanc_node in sanctioned_nodes[:config.EGO_SUBGRAPH_MAX_SAMPLES]:
    ego_edges_df, ego_nodes = extract_ego_subgraph_df(df_edges, sanc_node, radius=config.EGO_SUBGRAPH_RADIUS)
    ego_graphs[sanc_node] = ego_edges_df
    ego_node_sets[sanc_node] = ego_nodes
    logger.info(f"    Node {sanc_node}: {len(ego_nodes)} nodes, {len(ego_edges_df)} edges")

# Static visualizations
logger.info("  Creating static visualizations...")
for idx, sanc_node in enumerate(list(ego_graphs.keys())[:config.EGO_SUBGRAPH_MAX_SAMPLES]):
    ego_edges_df = ego_graphs[sanc_node]
    ego_nodes = ego_node_sets[sanc_node]
    
    # Build networkx graph from ego edges (on-the-fly, not full graph)
    G_ego = nx.DiGraph()
    for _, row in ego_edges_df.iterrows():
        from_id = int(row['from_id'])
        to_id = int(row['to_id'])
        weight = row['Amount Sent']
        G_ego.add_edge(from_id, to_id, weight=weight, timestamp=row['Timestamp'])
    
    fig, ax = plt.subplots(figsize=(14, 11))
    pos = nx.spring_layout(G_ego, k=2, iterations=100, seed=42)

    node_colors = ['darkred' if node == sanc_node else 'steelblue'
                   for node in G_ego.nodes()]
    node_sizes = [400 if node == sanc_node
                  else 50 + G_ego.in_degree(node) * 10
                  for node in G_ego.nodes()]

    nx.draw_networkx_nodes(G_ego, pos, node_color=node_colors, node_size=node_sizes,
                          alpha=0.85, ax=ax, edgecolors='black', linewidths=1.5)

    edges = G_ego.edges()
    edge_widths = [G_ego[u][v]['weight'] / 50 for u, v in edges]
    nx.draw_networkx_edges(G_ego, pos, edge_color='gray', arrows=True, arrowsize=15,
                          connectionstyle='arc3,rad=0.1', width=edge_widths, alpha=0.5, ax=ax)

    sanctioned_labels = {sanc_node: f"SANCTIONED\nNODE\n{sanc_node}"}
    nx.draw_networkx_labels(G_ego, pos, labels=sanctioned_labels,
                           font_size=9, font_weight='bold', ax=ax)

    ax.set_title(f'Ego-Centric Subgraph (Radius={config.EGO_SUBGRAPH_RADIUS}) Around OFAC Node {sanc_node}\n'
                f'Nodes: {len(G_ego.nodes())}, Edges: {len(G_ego.edges())}',
                fontsize=13, fontweight='bold', pad=15)
    ax.axis('off')
    plt.tight_layout()
    
    output_path = config.OUTPUT_DIR / f'viz2a_ego_subgraph_static_{idx}_{sanc_node}.png'
    fig.savefig(output_path, dpi=config.DPI, bbox_inches='tight')
    logger.info(f"  ✓ Saved: {output_path}")
    plt.close()

# Interactive visualizations
logger.info("  Creating interactive visualizations (PyVis)...")
for idx, sanc_node in enumerate(list(ego_graphs.keys())[:config.EGO_SUBGRAPH_MAX_SAMPLES]):
    ego_edges_df = ego_graphs[sanc_node]
    ego_nodes = ego_node_sets[sanc_node]
    
    net = Network(directed=True, height='800px', width='100%', notebook=False)
    net.physics.enabled = True
    net.physics.stabilization.iterations = 200

    for node in ego_nodes:
        is_sanctioned = (node == sanc_node)
        color = 'red' if is_sanctioned else 'steelblue'
        size = 50 if is_sanctioned else 30
        title = f"{'[SANCTIONED] ' if is_sanctioned else ''}Node {node}"
        net.add_node(node, label=str(node), color=color, size=size, title=title)

    for _, row in ego_edges_df.iterrows():
        u = int(row['from_id'])
        v = int(row['to_id'])
        weight = row['Amount Sent']
        net.add_edge(u, v, value=weight, title=f"{weight:.2f} ETH")

    output_file = config.OUTPUT_DIR / f'viz2b_ego_subgraph_interactive_{idx}_{sanc_node}.html'
    net.show(str(output_file))
    logger.info(f"  ✓ Saved: {output_file}")

## Visualization 3: Temporal Evolution Snapshots (Production)

In [ ]:
logger.info("\n[VIZ 3] Temporal Evolution Snapshots")

# Create temporal windows
SECONDS_PER_WEEK = 7 * 86400
df_edges['week'] = (df_edges['Timestamp'] // SECONDS_PER_WEEK).astype(int)

weeks = sorted(df_edges['week'].unique())
window_indices = np.linspace(0, len(weeks) - 1, config.TEMPORAL_WINDOWS, dtype=int)
selected_weeks = [weeks[i] for i in window_indices]

logger.info(f"  Total weeks: {len(weeks)}, Selected: {config.TEMPORAL_WINDOWS}")

fig, axes = plt.subplots(2, 5, figsize=(26, 11))
axes = axes.flatten()

sanctioned_set = set(df_labels[df_labels['is_sanctioned'] == 1]['node_id'].values)

for plot_idx, week in enumerate(selected_weeks):
    ax = axes[plot_idx]
    logger.info(f"  Processing week {week} ({plot_idx + 1}/{config.TEMPORAL_WINDOWS})...")

    week_edges = df_edges[df_edges['week'] == week]
    if len(week_edges) == 0:
        ax.text(0.5, 0.5, 'No transactions', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'Week {week}\n(0 edges)')
        ax.axis('off')
        continue

    # Build graph for this week
    G_week = nx.DiGraph()
    for _, row in week_edges.iterrows():
        from_id, to_id, amount = int(row['from_id']), int(row['to_id']), row['Amount Sent']
        G_week.add_edge(from_id, to_id, weight=amount)

    if len(G_week.nodes()) > 0:
        largest_cc = max(nx.weakly_connected_components(G_week), key=len)
        G_week_viz = G_week.subgraph(largest_cc).copy()
    else:
        G_week_viz = G_week

    pos = nx.spring_layout(G_week_viz, k=0.5, iterations=20, seed=42)

    node_sizes = [300 if node in sanctioned_set else (10 + G_week_viz.degree(node) * 5)
                  for node in G_week_viz.nodes()]
    node_colors = ['darkred' if node in sanctioned_set else 'steelblue'
                   for node in G_week_viz.nodes()]

    nx.draw_networkx_nodes(G_week_viz, pos, node_size=node_sizes, node_color=node_colors,
                          alpha=0.7, ax=ax)
    nx.draw_networkx_edges(G_week_viz, pos, edge_color='gray', arrows=False, alpha=0.2, ax=ax)

    ax.set_title(f'Week {week}\n({len(G_week_viz.nodes())} nodes, {len(G_week_viz.edges())} edges)',
                fontsize=10, fontweight='bold')
    ax.axis('off')

for idx in range(len(selected_weeks), len(axes)):
    fig.delaxes(axes[idx])

fig.suptitle('Temporal Evolution of Transaction Graph (10 snapshots)\n'
            'Red stars = Sanctioned addresses; Blue dots = Benign addresses; Size ∝ Degree',
            fontsize=14, fontweight='bold', y=0.995)

plt.tight_layout()
output_path = config.OUTPUT_DIR / 'viz3_temporal_evolution_grid.png'
fig.savefig(output_path, dpi=config.DPI, bbox_inches='tight')
logger.info(f"  ✓ Saved: {output_path}")
plt.close()

## Visualization 5: Evaluation Metrics (Production)

In [ ]:
logger.info("\n[VIZ 5] Evaluation Metrics (Confusion Matrix & Curves)")

# Get validation predictions
logger.info("  Computing validation predictions...")
with torch.no_grad():
    logits = model(data.x, data.edge_index, data.edge_attr)
    val_logits = logits[data.val_mask]

y_true = data.y[data.val_mask].cpu().numpy()
y_proba = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
y_pred = (y_proba >= 0.5).astype(int)

logger.info(f"  Predictions: {(y_pred == 1).sum()} positive, {(y_pred == 0).sum()} negative")

# Confusion Matrix
logger.info("  Creating confusion matrix...")
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Sanctioned', 'Sanctioned'],
            yticklabels=['Non-Sanctioned', 'Sanctioned'],
            cbar_kws={'label': 'Count'},
            annot_kws={'size': 14, 'weight': 'bold'}, ax=ax)
ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_title('Confusion Matrix (Validation Set)\nThreshold = 0.5',
            fontsize=13, fontweight='bold', pad=15)

tn, fp, fn, tp = cm.ravel()
metrics_text = f'TP={tp}, FN={fn}\nFP={fp}, TN={tn}\nPrec={tp/(tp+fp) if tp+fp>0 else 0:.3f}, Rec={tp/(tp+fn) if tp+fn>0 else 0:.3f}'
ax.text(0.5, -0.18, metrics_text, transform=ax.transAxes, ha='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
path = config.OUTPUT_DIR / 'viz5a_confusion_matrix.png'
fig.savefig(path, dpi=config.DPI, bbox_inches='tight')
logger.info(f"  ✓ Saved: {path}")
plt.close()

# ROC Curve
logger.info("  Creating ROC curve...")
fpr, tpr, _ = roc_curve(y_true, y_proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(fpr, tpr, color='darkorange', lw=3, label=f'ROC Curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
ax.fill_between(fpr, tpr, alpha=0.2, color='darkorange')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('ROC Curve', fontsize=13, fontweight='bold', pad=15)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
path = config.OUTPUT_DIR / 'viz5b_roc_curve.png'
fig.savefig(path, dpi=config.DPI, bbox_inches='tight')
logger.info(f"  ✓ Saved: {path}")
plt.close()

# PR Curve
logger.info("  Creating Precision-Recall curve...")
precision, recall, _ = precision_recall_curve(y_true, y_proba)
pr_auc = auc(recall, precision)
baseline_precision = (y_true == 1).sum() / len(y_true)

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(recall, precision, color='green', lw=3, label=f'PR Curve (AUC = {pr_auc:.3f})')
ax.fill_between(recall, precision, alpha=0.2, color='green')
ax.axhline(y=baseline_precision, color='red', linestyle='--', lw=2,
          label=f'Baseline ({baseline_precision:.3f})')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel('Recall', fontsize=12, fontweight='bold')
ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
ax.set_title('Precision-Recall Curve (Primary for Imbalanced Data)',
            fontsize=13, fontweight='bold', pad=15)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
path = config.OUTPUT_DIR / 'viz5c_precision_recall_curve.png'
fig.savefig(path, dpi=config.DPI, bbox_inches='tight')
logger.info(f"  ✓ Saved: {path}")
plt.close()

## Completion Summary

In [ ]:
logger.info("\n" + "="*80)
logger.info("VISUALIZATIONS COMPLETE!")
logger.info("="*80)
logger.info(f"\nOutput Directory: {config.OUTPUT_DIR}")
logger.info("\nGenerated Files:")
logger.info("  ✓ viz1_embedding_projection_umap.png")
logger.info("  ✓ viz2a_ego_subgraph_static_*.png")
logger.info("  ✓ viz2b_ego_subgraph_interactive_*.html")
logger.info("  ✓ viz3_temporal_evolution_grid.png")
logger.info("  ✓ viz5a_confusion_matrix.png")
logger.info("  ✓ viz5b_roc_curve.png")
logger.info("  ✓ viz5c_precision_recall_curve.png")
logger.info("\n" + "="*80)